In [9]:
# ── Google Drive folder ID (from the share link) ──────────────────────────────
DRIVE_FOLDER_ID = '1ktyrAnzQJxG-xOlgfJr70RpJQR2xwB4u'

# ── Where to write all clips (inside your Drive so results are persisted) ─────
# Will be created if it does not exist.
OUTPUT_CLIPS_DIR = '/content/drive/Shareddrives/DMD_clips'

# ── Repo ──────────────────────────────────────────────────────────────────────
REPO_DIR = '/content/Driving_Distraction_Detection'
REPO_URL = 'https://github.com/AnnikaUnmuessig/Driving_Distraction_Detection.git'

# ── Behaviour ─────────────────────────────────────────────────────────────────
SKIP_UNCLASSIFIED = False   # set True to drop 'unclassified' clips
DROP_KEYWORDS     = ('hand', 'head', 'face', 'gaze', 'eye', 'eyes', 'mirror')

print('Configuration loaded ✓')

Configuration loaded ✓


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')

# Authenticate so we can call the Drive API
auth.authenticate_user()
print('Drive mounted and authenticated ✓')

In [4]:
#Extract tar files 
import os
import tarfile
from pathlib import Path

dmd_root = "/Users/annikaunmuessig/Developer/University/Group_Projects/Project_Multimodal/Driving_Distraction_Detection/DMD"

# Check if folder exists
if os.path.isdir(dmd_root):
    print(f'✓ Folder "{dmd_root}" exists')
    
    # Find and extract all .tar files
    data_path = Path(dmd_root)
    tar_files = list(data_path.glob('**/*.tar'))
    
    if tar_files:
        print(f'Found {len(tar_files)} .tar files to extract...')
        for tar_path in tar_files:
            print(f'  Extracting {tar_path.name}...')
            with tarfile.open(tar_path, 'r') as tar:
                tar.extractall(path=tar_path.parent)
        print('Extraction complete ✓')
    else:
        print('No .tar files found – proceeding with existing files.')
else:
    print(f'✗ Folder "{dmd_root}" does not exist')
    print('Please ensure the DMD folder is in the notebook directory')

✓ Folder "/Users/annikaunmuessig/Developer/University/Group_Projects/Project_Multimodal/Driving_Distraction_Detection/DMD" exists
Found 6 .tar files to extract...
  Extracting dmd-dataset-rgb_ir-gE-26.tar...
  Extracting dmd-dataset-rgb_ir-gC-11.tar...
  Extracting dmd-dataset-rgb_ir-gC-12.tar...
  Extracting dmd-dataset-rgb_ir-gC-13.tar...
  Extracting dmd-dataset-rgb_ir-gC-14.tar...
  Extracting dmd-dataset-rgb_ir-gC-15.tar...
Extraction complete ✓


In [10]:
%pip install -q opencv-python tqdm

import sys, os
repo_path = Path(REPO_DIR)
if repo_path.exists():
    print('Repo exists – pulling …')
    os.system(f'git -C "{REPO_DIR}" pull --ff-only')
else:
    print('Cloning repo …')
    os.system(f'git clone "{REPO_URL}" "{REPO_DIR}"')

scripts_path = str(repo_path / 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

print('Dependencies ready ✓')

You should consider upgrading via the '/Users/annikaunmuessig/Developer/University/Group_Projects/Project_Multimodal/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Cloning repo …
Dependencies ready ✓


fatal: could not create leading directories of '/content/Driving_Distraction_Detection': Read-only file system


In [ ]:
from __future__ import annotations

import json, re
from typing import Any

import cv2
from tqdm.notebook import tqdm

from clean_dmd_body_actions import clean_dmd_json  # from cloned repo

DMD_BODY_ACTION_MAP: dict[str, int] = {
    'safe_driving':         0,
    'texting_right':        1,
    'phonecall_right':      2,
    'texting_left':         3,
    'phonecall_left':       4,
    'radio':                5,
    'drinking':             6,
    'reach_side':           7,
    'hair_and_makeup':      8,
    'talking_to_passenger': 9,
    'reach_backseat':       10,
    'change_gear':          11,
    'stand_still_waiting':  12,
    'unclassified':         13,
}

print(f'Imports OK ✓  ({len(DMD_BODY_ACTION_MAP)} classes)')

Imports OK ✓  (14 classes)


In [13]:
_VIDEO_STEM_RE = re.compile(r'^(.+)_rgb_body$')


def find_leaf_folders(root: Path) -> list[Path]:
    """
    Recursively walk *root* and return every folder that contains at least
    one '*_rgb_body.mp4' file (= a 'base case' ready for processing).
    """
    leaves: list[Path] = []
    for folder in sorted(root.rglob('*')):
        if not folder.is_dir():
            continue
        # A leaf folder contains at least one body-camera video
        videos = [f for f in folder.iterdir()
                  if f.suffix == '.mp4' and _VIDEO_STEM_RE.match(f.stem)]
        if videos:
            leaves.append(folder)
    return leaves


def find_pairs_in_folder(folder: Path) -> list[tuple[Path, Path]]:
    """
    Inside a leaf folder, pair each '*_rgb_body.mp4' with its
    '*_rgb_ann_distraction_body_only.json' (already cleaned).
    """
    pairs: list[tuple[Path, Path]] = []
    for mp4 in sorted(folder.glob('*.mp4')):
        m = _VIDEO_STEM_RE.match(mp4.stem)
        if not m:
            continue
        prefix    = m.group(1)
        json_path = folder / f'{prefix}_rgb_ann_distraction_body_only.json'
        if json_path.exists():
            pairs.append((mp4, json_path))
        else:
            print(f'  [WARN] No body_only JSON for {mp4.name}')
    return pairs


def load_actions(json_path: Path) -> list[dict[str, Any]]:
    with json_path.open('r', encoding='utf-8') as fh:
        payload = json.load(fh)
    actions_raw = payload.get('openlabel', {}).get('actions', {})
    actions: list[dict[str, Any]] = []
    for action_id, action_data in actions_raw.items():
        atype = str(action_data.get('type', 'unclassified')).lower()
        for iv in action_data.get('frame_intervals', []):
            actions.append({
                'id': action_id, 'type': atype,
                'frame_start': int(iv['frame_start']),
                'frame_end':   int(iv['frame_end']),
            })
    actions.sort(key=lambda a: (a['frame_start'], a['frame_end']))
    return actions


def write_clip(cap, out_path: Path, fs: int, fe: int,
               fps: float, w: int, h: int) -> bool:
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))
    cap.set(cv2.CAP_PROP_POS_FRAMES, fs)
    written = 0
    for _ in range(fe - fs + 1):
        ret, frame = cap.read()
        if not ret:
            break
        writer.write(frame)
        written += 1
    writer.release()
    if written == 0:
        out_path.unlink(missing_ok=True)
        return False
    return True


def sanitize(s: str) -> str:
    return re.sub(r'[^a-zA-Z0-9_\-]', '_', s)


print('Helpers defined ✓')

Helpers defined ✓


In [ ]:
DATA_ROOT = Path('/Users/annikaunmuessig/Developer/University/Group_Projects/Project_Multimodal/Driving_Distraction_Detection/DMD/dmd')
clips_dir = Path(OUTPUT_CLIPS_DIR)
clips_dir.mkdir(parents=True, exist_ok=True)
for cls in DMD_BODY_ACTION_MAP:
    (clips_dir / cls).mkdir(exist_ok=True)

print(f'Output dir: {clips_dir}')
print('Class subfolders created ✓')

# Find all leaf folders
leaf_folders = find_leaf_folders(DATA_ROOT)
print(f'\nLeaf folders found: {len(leaf_folders)}')
for lf in leaf_folders:
    print(f'  {lf.relative_to(DATA_ROOT)}')

drop_kws    = tuple(k.lower() for k in DROP_KEYWORDS)
label_lines: list[str]  = []
stats: dict[str, int]   = {k: 0 for k in DMD_BODY_ACTION_MAP}

for leaf in tqdm(leaf_folders, desc='Leaf folders', unit='folder'):
    print(f'\n📂 {leaf.relative_to(DATA_ROOT)}')

    # ── Step A: clean raw annotation JSONs in this folder ────────────────────
    raw_jsons = [
        p for p in leaf.glob('*.json')
        if 'ann_distraction' in p.stem and '_body_only' not in p.stem
    ]
    for raw_json in raw_jsons:
        out_json = raw_json.with_name(f'{raw_json.stem}_body_only.json')
        if out_json.exists():
            print(f'  [SKIP clean] {out_json.name}')
            continue
        result   = clean_dmd_json(raw_json, out_json, drop_kws)
        n_act    = len(result.get('openlabel', {}).get('actions', {}))
        print(f'  ✓ cleaned {out_json.name}  ({n_act} body actions)')

    # ── Step B: find video–annotation pairs ──────────────────────────────────
    pairs = find_pairs_in_folder(leaf)
    if not pairs:
        print('  [WARN] No video–annotation pairs found, skipping.')
        continue

    # ── Step C: cut each video ───────────────────────────────────────────────
    for video_path, json_path in pairs:
        actions = load_actions(json_path)
        if not actions:
            print(f'  [WARN] No actions in {json_path.name}')
            continue

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print(f'  [ERROR] Cannot open {video_path.name}')
            continue

        fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
        w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f'  ✂  {video_path.name}  [{total} frames | {fps:.1f} fps | {w}×{h}]')

        for idx, action in enumerate(
            tqdm(actions, desc='    clips', unit='clip', leave=False)
        ):
            label_str = action['type']
            class_id  = DMD_BODY_ACTION_MAP.get(label_str, 13)  # 13 = unclassified

            if SKIP_UNCLASSIFIED and class_id == 13:
                continue

            fs = max(0, action['frame_start'])
            fe = min(total - 1, action['frame_end'])
            if fs > fe:
                continue

            clip_name = f'{video_path.stem}_clip{idx:04d}_{sanitize(label_str)}.mp4'
            out_path  = clips_dir / label_str / clip_name

            if out_path.exists():
                # Re-running: skip write but still register label
                label_lines.append(f'{label_str}/{clip_name} {class_id}')
                stats[label_str] += 1
                continue

            if write_clip(cap, out_path, fs, fe, fps, w, h):
                label_lines.append(f'{label_str}/{clip_name} {class_id}')
                stats[label_str] += 1

        cap.release()

print(f'\n✅ Done — {len(label_lines)} clips total')